In [4]:
%pip install -q pandas scikit-learn streamlit joblib requests

import pandas as pd
import sklearn
import streamlit
import joblib
import requests

print(f"pandas:       {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"streamlit:    {streamlit.__version__}")
print(f"joblib:       {joblib.__version__}")
print(f"requests:     {requests.__version__}")

Note: you may need to restart the kernel to use updated packages.
pandas:       3.0.3
scikit-learn: 1.9.0
streamlit:    1.58.0
joblib:       1.5.3
requests:     2.34.2



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from pathlib import Path
import requests
import pandas as pd

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

csv_path = data_dir / "results.csv"

# Use the raw file URL
url = "https://raw.githubusercontent.com/IBM-SkillsBuild-AI-Builders-Challenge/hands-on-labs/main/02_football_lab_june/04_data/results.csv"

# Force re-download (previous download got HTML)
print(f"Downloading {url} ...")
r = requests.get(url, timeout=30)
r.raise_for_status()
csv_path.write_bytes(r.content)
print(f"Saved {csv_path.stat().st_size / 1024:.1f} KB to {csv_path}")

matches = pd.read_csv(csv_path, parse_dates=["date"])
print(f"Shape: {matches.shape}")
print(f"Date range: {matches['date'].min().date()} → {matches['date'].max().date()}")
matches.head(3)

Saved 3650.0 KB to data\results.csv
Shape: (49329, 9)
Date range: 1872-11-30 → 2026-06-27


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False


In [6]:
print("Top 10 tournament types:")
print(matches["tournament"].value_counts().head(10).to_string())

print("\nTop 15 teams by matches played:")
all_teams = pd.concat([matches["home_team"], matches["away_team"]])
print(all_teams.value_counts().head(15).to_string())

print("\nMatches per decade:")
decades = (matches["date"].dt.year // 10 * 10).astype(int)
print(decades.value_counts().sort_index().to_string())

Top 10 tournament types:
tournament
Friendly                                18257
Soccer World Cup qualification           8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
Soccer World Cup                         1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620

Top 15 teams by matches played:
Sweden         1102
England        1091
Argentina      1069
Brazil         1060
Germany        1032
South Korea    1008
Hungary        1004
Mexico         1003
Uruguay         973
France          936
Italy           891
Poland          890
Switzerland     885
Netherlands     880
Norway          873

Matches per decade:
date
1870      13
1880      55
1890      59
1900     137
1910     330
1920     831
1930    1079
1940     833
1950    1651
1960    2971
1970    4133
19

In [7]:
from collections import defaultdict

major_tournaments = {
    "Soccer World Cup",
    "Soccer World Cup qualification",
    "UEFA Euro",
    "UEFA Euro qualification",
    "Copa América",
    "African Cup of Nations",
}

m = matches[matches["date"] >= "1990-01-01"].sort_values("date").reset_index(drop=True)

team_history = defaultdict(list)

rows = []
for _, row in m.iterrows():
    home, away = row["home_team"], row["away_team"]
    h_hist = team_history[home]
    a_hist = team_history[away]

    def winrate(hist):
        return sum(1 for _, _, w in hist if w == 1) / len(hist) if hist else 0.5

    def goal_avg(hist):
        return sum(gf for gf, _, _ in hist) / len(hist) if hist else 1.0

    def recent_form(hist):
        last = hist[-10:]
        return sum(1 for _, _, w in last if w == 1) / len(last) if len(last) == 10 else 0.5

    if row["home_score"] > row["away_score"]:
        outcome, home_won, away_won = 0, 1, 0
    elif row["home_score"] < row["away_score"]:
        outcome, home_won, away_won = 2, 0, 1
    else:
        outcome, home_won, away_won = 1, 0, 0

    rows.append({
        "date": row["date"],
        "home_team": home,
        "away_team": away,
        "team_a_winrate": winrate(h_hist),
        "team_b_winrate": winrate(a_hist),
        "team_a_goal_avg": goal_avg(h_hist),
        "team_b_goal_avg": goal_avg(a_hist),
        "team_a_recent_form": recent_form(h_hist),
        "team_b_recent_form": recent_form(a_hist),
        "is_neutral": int(row["neutral"]),
        "is_major_tournament": 1 if row["tournament"] in major_tournaments else 0,
        "outcome": outcome,
    })

    team_history[home].append((row["home_score"], row["away_score"], home_won))
    team_history[away].append((row["away_score"], row["home_score"], away_won))

features_df = pd.DataFrame(rows)
print(f"Shape: {features_df.shape}")
features_df.head(3)

Shape: (32212, 12)


,date,home_team,away_team,team_a_winrate,team_b_winrate,team_a_goal_avg,team_b_goal_avg,team_a_recent_form,team_b_recent_form,is_neutral,is_major_tournament,outcome
0,1990-01-12,Algeria,Mali,0.5,0.5,1.0,1.0,0.5,0.5,1,0,0
1,1990-01-14,Algeria,Cameroon,1.0,0.5,5.0,1.0,0.5,0.5,1,0,0
2,1990-01-17,Greece,Belgium,0.5,0.5,1.0,1.0,0.5,0.5,0,0,0


In [8]:
feature_cols = [
    "team_a_winrate", "team_b_winrate",
    "team_a_goal_avg", "team_b_goal_avg",
    "team_a_recent_form", "team_b_recent_form",
    "is_neutral", "is_major_tournament",
]

cutoff = pd.Timestamp("2018-01-01")
train_mask = features_df["date"] < cutoff
test_mask = features_df["date"] >= cutoff

X_train = features_df.loc[train_mask, feature_cols]
X_test = features_df.loc[test_mask, feature_cols]
y_train = features_df.loc[train_mask, "outcome"]
y_test = features_df.loc[test_mask, "outcome"]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")
print("\nClass distribution in y_train:")
print(y_train.value_counts(normalize=True).sort_index().round(3).to_string())

X_train: (24179, 8), X_test: (8033, 8)
y_train: (24179,), y_test: (8033,)

Class distribution in y_train:
outcome
0    0.487
1    0.237
2    0.276


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
majority_class = y_train.value_counts().idxmax()
baseline = (y_test == majority_class).mean()

print(f"Test accuracy: {acc*100:.2f}%")
print(f"Baseline (always predict majority class): {baseline*100:.2f}%")

print("\nConfusion matrix (rows=actual, cols=predicted):")
labels = ["Home win", "Draw", "Away win"]
cm = confusion_matrix(y_test, y_pred)
print(f"{'':<10}" + "".join(f"{l:>10}" for l in labels))
for label, row in zip(labels, cm):
    print(f"{label:<10}" + "".join(f"{v:>10}" for v in row))

print("\nFeature importances:")
for name, imp in sorted(zip(feature_cols, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {name:<22} {imp:.4f}")

Test accuracy: 55.76%
Baseline (always predict majority class): 47.19%

Confusion matrix (rows=actual, cols=predicted):
            Home win      Draw  Away win
Home win        3393        32       366
Draw            1445        31       434
Away win        1222        55      1055

Feature importances:
  team_b_winrate         0.2211
  team_a_winrate         0.2056
  team_b_goal_avg        0.1882
  team_a_goal_avg        0.1826
  team_b_recent_form     0.0752
  team_a_recent_form     0.0751
  is_neutral             0.0274
  is_major_tournament    0.0248


In [10]:
from pathlib import Path
import joblib

models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

wc_qual = matches[matches["tournament"] == "Soccer World Cup qualification"]
soccer_teams = set(pd.concat([wc_qual["home_team"], wc_qual["away_team"]]).unique())

all_teams = pd.concat([matches["home_team"], matches["away_team"]]).unique()
team_stats = {}

for team in all_teams:
    if team not in soccer_teams:
        continue

    home = matches[matches["home_team"] == team]
    away = matches[matches["away_team"] == team]
    total = len(home) + len(away)
    if total < 30:
        continue

    home_wins = int((home["home_score"] > home["away_score"]).sum())
    away_wins = int((away["away_score"] > away["home_score"]).sum())
    total_goals = int(home["home_score"].sum() + away["away_score"].sum())

    last10 = matches[(matches["home_team"] == team) | (matches["away_team"] == team)] \
        .sort_values("date").tail(10)
    if len(last10) == 10:
        wins = 0
        for _, r in last10.iterrows():
            if r["home_team"] == team and r["home_score"] > r["away_score"]:
                wins += 1
            elif r["away_team"] == team and r["away_score"] > r["home_score"]:
                wins += 1
        recent_form = wins / 10
    else:
        recent_form = 0.5

    team_stats[team] = {
        "winrate": (home_wins + away_wins) / total,
        "goal_avg": total_goals / total,
        "recent_form": recent_form,
        "matches_played": total,
    }

joblib.dump(model, models_dir / "match_predictor.pkl")
joblib.dump({"team_stats": team_stats, "feature_cols": feature_cols},
            models_dir / "team_data.pkl")

print(f"Saved model and team_data for {len(team_stats)} teams.")
print("\nTop 5 by winrate (≥100 matches):")
eligible = [(t, s) for t, s in team_stats.items() if s["matches_played"] >= 100]
for team, s in sorted(eligible, key=lambda x: -x[1]["winrate"])[:5]:
    print(f"  {team:<25} winrate={s['winrate']:.3f}  matches={s['matches_played']}")

Saved model and team_data for 215 teams.

Top 5 by winrate (≥100 matches):
  Brazil                    winrate=0.632  matches=1060
  Spain                     winrate=0.587  matches=784
  Germany                   winrate=0.578  matches=1032
  England                   winrate=0.571  matches=1091
  Iran                      winrate=0.566  matches=613
